# External mapper deep dive (marketing appendix; not Fig 4d)

This notebook complements `idtrack/reproducibility/experiments/experiment_tool_comparison/00_tool_comparison_capability_matrix_fig4d.ipynb` by generating **extra** figures/tables that help you market IDTrack.

## Rationale

External identifier mappers (BioMart, MyGene.info, g:Profiler, gget) are extremely useful, but they typically behave as **point-in-time services**.
This notebook is an evidence pack for two practical consequences you can show in a Results section without overclaiming:

- **Outcome semantics**: how often queries become 1→0 / 1→1 / 1→n under different backends and scenarios (HGNC↔Ensembl, UniProt↔Ensembl).
- **Cross-method variability**: how often different backends disagree on the returned *set* of targets (quantified via Jaccard similarities).

This is *not* an accuracy benchmark; it is a marketing-friendly demonstration that “mapping” is not a single deterministic function in practice, and that reproducibility requires explicit semantics and snapshot controls.

## Outputs (optional marketing pool)

- `_outputs/_publication/figures/fig_external_mapper_outcome_profiles.pdf`
- `_outputs/_publication/figures/fig_external_mapper_agreement_heatmaps.pdf`
- `_outputs/_publication/figures/fig_external_mapper_output_sizes.pdf`
- `_outputs/_publication/figures/fig_external_mapper_jaccard_distributions.pdf`
- `_outputs/_publication/figures/fig_external_mapper_consensus_rate.pdf`
- `_outputs/_publication/tables/external_mapper_consensus_rates.csv`
- `idtrack/docs/_notebooks/idtrack_cache/experiments/comparison/external_mapper_demo_outputs.csv`
- `idtrack/docs/_notebooks/idtrack_cache/experiments/comparison/external_mapper_disagreements.csv`

## Caching

- Uses the same cache directory as `experiment_tool_comparison/00_tool_comparison_capability_matrix_fig4d.ipynb`.
- If caches are missing, the notebook will query external APIs once and write pickles under the shared cache.
- Re-running should be fast and should not hit public APIs unless you delete cache files.


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import os
import sys

# Add experiments/src to sys.path (layout-aware; works on Slurm and locally)
REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).expanduser().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (
    (REPO_ROOT / 'idtrack').is_dir()
    and ((REPO_ROOT / 'reproducibility').is_dir() or (REPO_ROOT / 'idtrack' / 'reproducibility').is_dir())
):
    REPO_ROOT = REPO_ROOT.parent

REPRO_ROOT = REPO_ROOT / 'reproducibility' if (REPO_ROOT / 'reproducibility').is_dir() else REPO_ROOT / 'idtrack' / 'reproducibility'
EXPERIMENTS_SRC = REPRO_ROOT / 'experiments' / 'src'
if str(EXPERIMENTS_SRC) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    build_tool_comparison_query_sets,
    MANUSCRIPT_COLORS,
    read_pickle,
    notebook_context,
    save_figure,
    safe_tag,
    write_pickle,
)

ctx = notebook_context('comparison', start=REPO_ROOT)
plt.rcParams.update({'savefig.dpi': 300})

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

SPECIES = 'human'

# Query cohorts (HGNC-derived; cached) — avoids hard-coded gene lists.
N_QUERIES = int(os.environ.get('TOOL_COMPARISON_N_QUERIES', '750'))
QUERY_SEED = int(os.environ.get('TOOL_COMPARISON_QUERY_SEED', '0'))
CHUNK_SIZE = int(os.environ.get('TOOL_COMPARISON_CHUNK_SIZE', '500'))
PAUSE_S = float(os.environ.get('TOOL_COMPARISON_PAUSE_S', '0.2'))

# IDTrack is optional in this notebook (graph loading can be memory intensive).
IDTRACK_SNAPSHOT_RELEASE = int(os.environ.get('IDTRACK_SNAPSHOT_RELEASE', '114'))
TOOL_COMPARISON_INCLUDE_IDTRACK = os.environ.get('TOOL_COMPARISON_INCLUDE_IDTRACK', '1').strip() in {'1', 'true', 'True'}

query_manifest = build_tool_comparison_query_sets(
    CACHE_DIR,
    start=REPO_ROOT,
    species=SPECIES,
    n=N_QUERIES,
    seed=QUERY_SEED,
    symbol_alias_fraction=0.25,
    symbol_prev_fraction=0.15,
    ensembl_versioned_fraction=0.01,
    include_negative_controls=True,
)

QUERY_ENSG = list(query_manifest['QUERY_ENSG'])
QUERY_SYMBOLS = list(query_manifest['QUERY_SYMBOLS'])
QUERY_UNIPROT = list(query_manifest['QUERY_UNIPROT'])
print('Query manifest:', query_manifest['meta'])

METHODS = ['pybiomart', 'mygene', 'gprofiler', 'gget']

PYBIOMART_RELEASE = 107

SCENARIOS = [
    {
        'label': 'Ensembl→HGNC',
        'ids': QUERY_ENSG,
        'input_db': 'ensembl_gene',
        'output_db': 'HGNC Symbol',
    },
    {
        'label': 'Ensembl→UniProt',
        'ids': QUERY_ENSG,
        'input_db': 'ensembl_gene',
        'output_db': 'UniProtKB/Swiss-Prot',
    },
    {
        'label': 'HGNC→Ensembl',
        'ids': QUERY_SYMBOLS,
        'input_db': 'HGNC Symbol',
        'output_db': 'ensembl_gene',
    },
    {
        'label': 'UniProt→Ensembl',
        'ids': QUERY_UNIPROT,
        'input_db': 'UniProtKB/Swiss-Prot',
        'output_db': 'ensembl_gene',
    },
]

print('Methods:', METHODS)
print('Scenarios:', [s['label'] for s in SCENARIOS])


In [ ]:
# -------------------- Compute / load cached external mappings --------------------

import hashlib

import idtrack._external_mappers as ext


def _ids_fingerprint(ids: list[str]) -> str:
    # Used only for cache file naming.
    h = hashlib.md5('|'.join(map(str, ids)).encode('utf-8')).hexdigest()  # noqa: S324
    return h[:12]


def _cache_path(label: str, method: str, input_db: str, output_db: str, ids: list[str]) -> Path:
    fp = _ids_fingerprint(ids)
    tag = (
        f"extmap_{safe_tag(label)}_{method}_in{safe_tag(input_db)}_out{safe_tag(output_db)}_"
        f"n{len(ids)}_{fp}_species{safe_tag(SPECIES)}_pybiomart{PYBIOMART_RELEASE}.pickle"
    )
    return CACHE_DIR / tag


def _idtrack_cache_path(label: str, input_db: str, output_db: str, ids: list[str]) -> Path:
    fp = _ids_fingerprint(ids)
    tag = (
        f"idtrackmap_{safe_tag(label)}_in{safe_tag(input_db)}_out{safe_tag(output_db)}_"
        f"n{len(ids)}_{fp}_species{safe_tag(SPECIES)}_snapshot{IDTRACK_SNAPSHOT_RELEASE}.pickle"
    )
    return CACHE_DIR / tag


def _normalize(df: pd.DataFrame | None) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=['input_id', 'output_id', 'mapping'])
    cols = [c for c in ['input_id', 'output_id', 'mapping', 'method', 'input_db', 'output_db', 'release_used'] if c in df.columns]
    return df[cols].copy()


results: dict[tuple[str, str], pd.DataFrame] = {}
errors: list[dict] = []

for scenario in SCENARIOS:
    label = str(scenario['label'])
    input_db = str(scenario['input_db'])
    output_db = str(scenario['output_db'])
    ids = [str(x) for x in (scenario['ids'] or [])]

    for method in METHODS:
        p = _cache_path(label, method, input_db, output_db, ids)
        if p.exists():
            results[(label, method)] = _normalize(read_pickle(p))
            continue

        try:
            kwargs = {
                'ids': ids,
                'input_db': input_db,
                'output_db': output_db,
                'method': method,
                'species': SPECIES,
                'chunk_size': int(CHUNK_SIZE),
                'pause': float(PAUSE_S),
                'verbose': 2,
            }
            if method == 'pybiomart':
                kwargs['release_for_pybiomart'] = PYBIOMART_RELEASE

            df = ext.convert_ids(**kwargs)
            df = _normalize(df)
            write_pickle(df, p)
            results[(label, method)] = df
        except Exception as e:
            errors.append(
                {
                    'scenario': label,
                    'input_db': input_db,
                    'output_db': output_db,
                    'method': method,
                    'error': repr(e),
                }
            )

print('Cached result blocks:', len(results))

# Load cached IDTrack conversions if available, so plots can include IDTrack without running it locally.
if TOOL_COMPARISON_INCLUDE_IDTRACK:
    loaded_any = False
    for scenario in SCENARIOS:
        label = str(scenario['label'])
        input_db = str(scenario['input_db'])
        output_db = str(scenario['output_db'])
        ids = [str(x) for x in (scenario['ids'] or [])]
        p = _idtrack_cache_path(label, input_db, output_db, ids)
        if p.exists():
            results[(label, 'IDTrack')] = _normalize(read_pickle(p))
            loaded_any = True
    if loaded_any and 'IDTrack' not in METHODS:
        METHODS = ['IDTrack'] + list(METHODS)
pd.DataFrame(errors) if errors else None


In [ ]:
# -------------------- Summaries (outcomes + per-query output sets) --------------------

def outputs_by_input(df: pd.DataFrame, inputs: list[str]) -> dict[str, set[str]]:
    if df is None or df.empty:
        return {str(i): set() for i in inputs}
    out: dict[str, set[str]] = {}
    for inp, sub in df.groupby('input_id'):
        vals = [v for v in sub['output_id'].tolist() if v is not None and str(v).strip() not in {'', 'nan', 'None', 'null'}]
        out[str(inp)] = set(map(str, vals))
    # ensure all inputs exist
    for i in inputs:
        out.setdefault(str(i), set())
    return out


def outcome_counts(df: pd.DataFrame, inputs: list[str]) -> dict[str, int]:
    if df is None or df.empty:
        return {'1→0': len(set(inputs)), '1→1': 0, '1→n': 0}
    per = df.drop_duplicates('input_id')
    vc = per['mapping'].value_counts().to_dict()
    return {
        '1→0': int(vc.get('1:0', 0)),
        '1→1': int(vc.get('1:1', 0)),
        '1→n': int(vc.get('1:n', 0)),
    }


summary_rows = []
output_sets: dict[tuple[str, str], dict[str, set[str]]] = {}

for scenario in SCENARIOS:
    label = str(scenario['label'])
    input_db = str(scenario['input_db'])
    output_db = str(scenario['output_db'])
    ids = [str(x) for x in (scenario['ids'] or [])]
    for method in METHODS:
        df = results.get((label, method), pd.DataFrame())
        summary_rows.append({'scenario': label, 'input_db': input_db, 'output_db': output_db, 'method': method, **outcome_counts(df, ids)})
        output_sets[(label, method)] = outputs_by_input(df, ids)

summary = pd.DataFrame(summary_rows)
if not summary.empty:
    summary['n_inputs'] = summary[['1→0', '1→1', '1→n']].sum(axis=1)
    summary['frac_1_to_0'] = summary['1→0'] / summary['n_inputs'].replace(0, pd.NA)
    summary['frac_1_to_1'] = summary['1→1'] / summary['n_inputs'].replace(0, pd.NA)
    summary['frac_1_to_n'] = summary['1→n'] / summary['n_inputs'].replace(0, pd.NA)
    summary = summary.sort_values(['scenario', 'output_db', 'method']).reset_index(drop=True)
summary


In [ ]:
# -------------------- Figure: outcome profiles (all scenarios) --------------------

if summary is None or summary.empty:
    print('No summary available; skipping outcome-profile figure.')
else:
    n = len(SCENARIOS)
    ncols = 2
    nrows = (n + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(12.5, 4.2 * nrows), constrained_layout=True)
    axes = list(np.ravel(axes))

    for ax, scenario in zip(axes, SCENARIOS, strict=False):
        label = str(scenario['label'])
        title = f"{label} ({scenario['input_db']} → {scenario['output_db']})"

        sub = summary[summary['scenario'] == label]
        if sub.empty:
            ax.axis('off')
            ax.text(0.5, 0.5, f'No rows for {label}', ha='center', va='center')
            continue

        s = sub.set_index('method')[['1→0', '1→1', '1→n']]
        s = s.reindex([m for m in METHODS if m in s.index])
        s = s.div(s.sum(axis=1), axis=0)
        s.plot(
            kind='bar',
            stacked=True,
            ax=ax,
            color=[MANUSCRIPT_COLORS['1→0'], MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']],
        )
        ax.set_ylim(0, 1)
        ax.set_ylabel('Fraction of queries')
        ax.set_title(title)
        ax.legend(['1→0', '1→1', '1→n'], loc='upper right', frameon=True)

    for ax in axes[n:]:
        ax.axis('off')

    written = save_figure(fig, 'fig_external_mapper_outcome_profiles.pdf', ctx, formats=('pdf',))
    print('Saved:', written['pdf'])

# -------------------- Figure: output set sizes (per method; per scenario) --------------------

size_rows = []
for scenario in SCENARIOS:
    label = str(scenario['label'])
    ids = [str(x) for x in (scenario['ids'] or [])]
    for method in METHODS:
        s = output_sets.get((label, method), {})
        for q in ids:
            size_rows.append({'scenario': label, 'method': method, 'input_id': str(q), 'n_outputs': int(len(s.get(str(q), set())))})

sizes = pd.DataFrame(size_rows)

if sizes.empty:
    print('No output-size rows; skipping output-size figure.')
else:
    n = len(SCENARIOS)
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    fig3, axes3 = plt.subplots(nrows, ncols, figsize=(12.5, 4.0 * nrows), constrained_layout=True)
    axes3 = list(np.ravel(axes3))

    for ax, scenario in zip(axes3, SCENARIOS, strict=False):
        label = str(scenario['label'])
        sub = sizes[sizes['scenario'] == label]
        if sub.empty:
            ax.axis('off')
            continue

        if sns is not None:
            sns.boxplot(data=sub, x='method', y='n_outputs', ax=ax, color=MANUSCRIPT_COLORS['1→1'])
            sns.stripplot(data=sub, x='method', y='n_outputs', ax=ax, color='#333333', alpha=0.6, size=3)
        else:
            groups = [sub[sub['method'] == m]['n_outputs'].tolist() for m in METHODS]
            ax.boxplot(groups, labels=METHODS)

        ax.set_title(f'Output set sizes: {label}')
        ax.set_xlabel('Method')
        ax.set_ylabel('# outputs per query')
        ax.set_ylim(bottom=0)
        ax.tick_params(axis='x', rotation=25)

    for ax in axes3[n:]:
        ax.axis('off')

    written3 = save_figure(fig3, 'fig_external_mapper_output_sizes.pdf', ctx, formats=('pdf',))
    print('Saved:', written3['pdf'])


In [ ]:
# -------------------- Figure: cross-method agreement heatmaps --------------------

def jaccard(a: set[str], b: set[str]) -> float:
    denom = len(a | b)
    return (len(a & b) / denom) if denom else 1.0


n = len(SCENARIOS)
ncols = 2
nrows = (n + ncols - 1) // ncols

fig2, axes2 = plt.subplots(nrows, ncols, figsize=(12.5, 5.0 * nrows), constrained_layout=True)
axes2 = list(np.ravel(axes2))

jaccard_rows = []

for ax, scenario in zip(axes2, SCENARIOS, strict=False):
    label = str(scenario['label'])
    ids = [str(x) for x in (scenario['ids'] or [])]

    # method x method: mean Jaccard across queries
    mat = pd.DataFrame(index=METHODS, columns=METHODS, dtype=float)
    for m1 in METHODS:
        for m2 in METHODS:
            s1 = output_sets.get((label, m1), {})
            s2 = output_sets.get((label, m2), {})
            vals = [jaccard(s1.get(str(q), set()), s2.get(str(q), set())) for q in ids]
            mat.loc[m1, m2] = float(np.mean(vals)) if vals else float('nan')

            # For distribution figure: only keep upper triangle (unordered pairs) and only if m1<m2
            if m1 < m2:
                for q, v in zip(ids, vals, strict=False):
                    jaccard_rows.append({'scenario': label, 'pair': f'{m1} vs {m2}', 'input_id': str(q), 'jaccard': float(v)})

    if sns is not None:
        sns.heatmap(mat, ax=ax, cmap='Blues', vmin=0, vmax=1, square=True, cbar=True)
    else:
        im = ax.imshow(mat.values, cmap='Blues', vmin=0, vmax=1)
        fig2.colorbar(im, ax=ax)
        ax.set_xticks(range(len(METHODS)))
        ax.set_yticks(range(len(METHODS)))
        ax.set_xticklabels(METHODS, rotation=30, ha='right')
        ax.set_yticklabels(METHODS)

    ax.set_title(f'Mean cross-method agreement: {label}')
    ax.set_xlabel('Method')
    ax.set_ylabel('Method')

for ax in axes2[n:]:
    ax.axis('off')

written2 = save_figure(fig2, 'fig_external_mapper_agreement_heatmaps.pdf', ctx, formats=('pdf',))
print('Saved:', written2['pdf'])

# -------------------- Figure: Jaccard distributions (variability) --------------------

jdf = pd.DataFrame(jaccard_rows)
if jdf.empty:
    print('No Jaccard rows; skipping distribution figure.')
else:
    fig4, ax4 = plt.subplots(1, 1, figsize=(10.5, 4.2))
    if sns is not None:
        sns.violinplot(data=jdf, x='scenario', y='jaccard', ax=ax4, inner='quartile', color=MANUSCRIPT_COLORS['1→1'])
    else:
        groups = [jdf[jdf['scenario'] == str(s['label'])]['jaccard'].tolist() for s in SCENARIOS]
        labels = [str(s['label']) for s in SCENARIOS]
        ax4.boxplot(groups, labels=labels)

    ax4.set_ylim(0, 1)
    ax4.set_ylabel('Pairwise Jaccard across methods (all unordered pairs, all queries)')
    ax4.set_xlabel('Scenario')
    ax4.set_title('External mapper variability (higher = more agreement)')
    ax4.tick_params(axis='x', rotation=15)
    fig4.tight_layout()

    written4 = save_figure(fig4, 'fig_external_mapper_jaccard_distributions.pdf', ctx, formats=('pdf',))
    print('Saved:', written4['pdf'])


In [ ]:
# -------------------- Export demo output table (CSV) --------------------

rows = []
for scenario in SCENARIOS:
    label = str(scenario['label'])
    input_db = str(scenario['input_db'])
    output_db = str(scenario['output_db'])
    ids = [str(x) for x in (scenario['ids'] or [])]

    for method in METHODS:
        s = output_sets.get((label, method), {})
        for q in ids:
            outs = sorted(s.get(str(q), set()))
            rows.append(
                {
                    'scenario': label,
                    'input_db': input_db,
                    'output_db': output_db,
                    'method': method,
                    'input_id': str(q),
                    'n_outputs': len(outs),
                    'outputs': ';'.join(outs[:30]),
                }
            )

demo = pd.DataFrame(rows)
out_csv = CACHE_DIR / 'external_mapper_demo_outputs.csv'
demo.to_csv(out_csv, index=False)
print('Wrote:', out_csv)

# Disagreement table (wide) for quick manuscript inspection
dis_rows = []
for scenario in SCENARIOS:
    label = str(scenario['label'])
    ids = [str(x) for x in (scenario['ids'] or [])]

    per_method = {m: output_sets.get((label, m), {}) for m in METHODS}

    for q in ids:
        sets = {m: set(per_method[m].get(str(q), set())) for m in METHODS}
        union = set().union(*sets.values()) if sets else set()
        vals = list(sets.values())
        inter = set(vals[0]).intersection(*vals[1:]) if vals else set()
        unique = {frozenset(v) for v in sets.values()}

        row = {
            'scenario': label,
            'input_id': str(q),
            'union_size': len(union),
            'intersection_size': len(inter),
            'n_unique_output_sets': len(unique),
            'all_agree': len(unique) == 1,
        }
        for m in METHODS:
            outs = sorted(sets[m])
            row[f'n_outputs_{m}'] = len(outs)
            row[f'outputs_{m}'] = ';'.join(outs[:30])
        dis_rows.append(row)

dis = pd.DataFrame(dis_rows)
out_dis = CACHE_DIR / 'external_mapper_disagreements.csv'
dis.to_csv(out_dis, index=False)
print('Wrote:', out_dis)

demo.head(10)


# Marketing extension: consensus vs disagreement rates

Agreement heatmaps show *where* methods differ; this section summarizes *how often* they differ.

For each scenario, we compute:

- **full consensus**: all methods return the same output set
- **any disagreement**: at least one method differs

This is a useful marketing artifact because it demonstrates that “external mapper” is not a single deterministic function.


In [ ]:
from experiments_utils import atomic_write_dataframe_csv  # noqa: E402

cons_rows = []
for scenario in SCENARIOS:
    label = str(scenario['label'])
    ids = [str(x) for x in (scenario['ids'] or [])]
    if not ids:
        continue
    per_method = {m: output_sets.get((label, m), {}) for m in METHODS}

    n_full = 0
    n_any = 0
    for q in ids:
        sets = [set(per_method[m].get(str(q), set())) for m in METHODS]
        if not sets:
            continue
        n_any += 1
        first = sets[0]
        if all(s == first for s in sets[1:]):
            n_full += 1

    cons_rows.append(
        {
            'scenario': label,
            'n_inputs': n_any,
            'n_full_consensus': n_full,
            'frac_full_consensus': n_full / n_any if n_any else float('nan'),
            'frac_any_disagreement': 1.0 - (n_full / n_any) if n_any else float('nan'),
        }
    )

cons = pd.DataFrame(cons_rows)
out_cons = ctx.manuscript_tables / 'external_mapper_consensus_rates.csv'
atomic_write_dataframe_csv(cons, out_cons, index=False)
atomic_write_dataframe_csv(cons, ctx.experiment_outputs / 'tables' / out_cons.name, index=False)
print('Wrote:', out_cons)
display(cons)

if not cons.empty:
    figC, axC = plt.subplots(1, 1, figsize=(7.2, 3.6), constrained_layout=True)
    axC.bar(cons['scenario'], cons['frac_full_consensus'], color=MANUSCRIPT_COLORS['1→1'])
    axC.set_ylim(0, 1)
    axC.set_ylabel('Full consensus fraction')
    axC.set_title('External mapper full-consensus rate (marketing summary)')
    axC.tick_params(axis='x', rotation=20)
    writtenC = save_figure(figC, 'fig_external_mapper_consensus_rate.pdf', ctx, formats=('pdf',))
    print('Saved:', writtenC['pdf'])
